In [1]:
import sys
from pathlib import Path


def sys_path_show():
    print(*sys.path, sep='\n')


def sys_path_append(p):
    p = Path(p)
    assert p.exists()
    p = str(p)
    None if p in sys.path else sys.path.append(p)


sys_path_append(Path("~/link").expanduser())
sys_path_append(Path("~/link/temp/241120_other_model/learn_TOSICA").expanduser())

In [2]:
import utils as ut
from utils.general import *
sc = ut.sc.sc

对于TOSICA本地化的修改

> 修改文件`TOSICA/__init__.py`

```python
# __version__ = '1.0.0'
# 就地导入，TOSICA并不存在于python3.9/site-packages
# version存放在上一级目录
from pathlib import Path
__version__ = Path(__file__).absolute().parent.parent.joinpath('VERSION.txt').read_text().replace('\n','')

```

> 修改文件 `TOSICA/train.py`

```python
def fit_model(adata, gmt_path, project = None,...):
    # ...

    # project_path = os.getcwd()+'/%s'%project
    # if os.path.exists(project_path) is False:
    #     os.makedirs(project_path)

    # 使用直接传入的路径
    from pathlib import Path
    project_path = Path(project).joinpath('TrainingModel')
    project_path.mkdir(parents=True,exist_ok=True)
    project = project_path.name
    project_path = str(project_path)

    # ...

```

> 修改文件 `TOSICA/pre.py`

```python

def prediect( #....
    # mask = np.load(mask_path)
    # project_path = os.getcwd()+'/%s'%project
    # pathway = pd.read_csv(project_path+'/pathway.csv', index_col=0)
    # dictionary = pd.read_table(project_path+'/label_dictionary.csv', sep=',',header=0,index_col=0)

    # 使用直接传入的路径 并读取 mask,pathway,dictionary
    from pathlib import Path
    project_path = Path(project)
    mask = np.load(project_path.joinpath('TrainingModel','mask.npy'))
    pathway = pd.read_csv(project_path.joinpath('TrainingModel','pathway.csv'), index_col=0)
    dictionary = pd.read_table(project_path.joinpath('TrainingModel',
                                                     'label_dictionary.csv'),
                               sep=',',header=0,index_col=0)
    project = project_path.name
    project_path = str(project_path)

```

In [3]:
from TOSICA import TOSICA

In [4]:
row = json.loads(Path(
    '~/link/other_model').expanduser().joinpath("parameters_run_cross_species_models.json").read_text())
display(row)

p_root_model = Path("~/link/temp/241120_other_model/learn_TOSICA").expanduser()

adata_ref = sc.read_h5ad(p_root_model.parent.joinpath('LCDCs_h.h5ad'))
adata_que = sc.read_h5ad(p_root_model.parent.joinpath('LCDCs_m.h5ad'))
p_gmt = p_root_model.joinpath('TOSICA/TOSICA/resources/GO_bp.gmt')

{'tissue': 'LC',
 'sp_ref': 'human',
 'path_ref': '/public/workspace/licanchengup/link/csMAHN_publish/cache/disease/LC_h_DendriticCells',
 'name_ref': 'LChDCs',
 'sp_simple_ref': 'h',
 'sp_que': 'mouse',
 'path_que': '/public/workspace/licanchengup/link/csMAHN_publish/cache/disease/LC_m_DendriticCells',
 'name_que': 'LCmDCs',
 'sp_simple_que': 'm',
 'key_cell_type': 'sub_cell_type'}

In [5]:
map_sp = {k: v for k, v in zip(
    'h,m,z,ma,c,x'.split(','),
    'human,mouse,zebrafish,macaque,chicken,xenopus'.split(',')
)}
map_sp_reverse = {v: k for k, v in map_sp.items()}

map_sp.update({k: v for k, v in zip(
    'hs,mm'.split(','),
    'human,mouse'.split(',')
)})

p_root = Path('~/link/csMAHN_publish').expanduser()
p_res = p_root.joinpath("res")
p_cache = p_root.joinpath("cache")


def get_path_varmap(
        sp_ref,
        sp_que,
        p_df_varmap=p_root.joinpath('homo/df_varmap.csv'),
        p_maps_SAMap=p_root.joinpath('homo/SAMap/maps_gene_name'),
        model='csMAHN'):
    """
    通过sp_ref 和 sp_que获取path_varmap
    ./homo/df_varmap.csv 存储了
    path_varmap路径及信息
"""
    p_df_varmap = Path(p_df_varmap)
    if model in 'TOSICA'.split(','):
        df_varmap = pd.read_csv(p_df_varmap)
        index_ = df_varmap.query(
            "sp_ref == '{}' & sp_que == '{}'".format(
                sp_ref, sp_que)).index
        assert index_.size == 1, "[get {} path]can not get speicifed and unique path\nsp_ref\tsp_que\n{}\t{}".format(
            index_.size, sp_ref, sp_que)
        res = Path(df_varmap.loc[index_[0], 'path'])
        if not res.is_absolute():
            res = p_df_varmap.parent.joinpath(res)
        assert res.exists(), "[not exists] {}".format(res)
        return res

    elif model == 'SAMap':
        return p_maps_SAMap
    else:
        raise Exception(
            "[Error] can not find path_varmap with model '{}'".format(model))


def get_1v1_matches(
        df_match,
        key_homology_type='homology_type',
        value_homology_type='ortholog_one2one'):
    """
    from came.pp.take_1v_matches
    """
    l, r = df_match.columns[:2]
    l_unique = df_match[l].value_counts(
    ).to_frame().query("count == 1").index
    r_unique = df_match[r].value_counts(
    ).to_frame().query("count == 1").index
    keep = pd.DataFrame({
        'l_is_unique': df_match[l].isin(l_unique),
        'r_is_unique': df_match[r].isin(r_unique)
    }).min(axis=1)
    df_match = df_match[keep]
    df_match = df_match.query(
        "{} == '{}'".format(
            key_homology_type,
            value_homology_type))
    return df_match


def df_varmap_query_exists(
        df_varmap,
        list_gn_ref=[],
        list_gn_que=[],
        model='both'):
    df_varmap = df_varmap.copy()
    df_varmap['gn_ref_exists'] = df_varmap['gn_ref'].isin(list_gn_ref)
    df_varmap['gn_que_exists'] = df_varmap['gn_que'].isin(list_gn_que)
    if model == 'both':
        df_varmap = df_varmap.query("gn_ref_exists & gn_que_exists")
    elif model == 'ref':
        df_varmap = df_varmap.query("gn_ref_exists")
    elif model == 'que':
        df_varmap = df_varmap.query("gn_que_exists")
    else:
        raise Exception('[Error] model must be one of both, ref, que')
    df_varmap = df_varmap.drop(
        columns='gn_ref_exists,gn_que_exists'.split(','))
    return df_varmap


df_homo = pd.read_csv(get_path_varmap(row['sp_ref'], row['sp_que'], model='TOSICA'),
                      names='gn_ref,gn_que,type'.split(','), skiprows=1).dropna(axis=0)
df_homo = get_1v1_matches(df_homo, 'type')
df_homo = df_varmap_query_exists(df_homo, adata_ref.var.index,
                                 adata_que.var.index, model='both')

print('[msg] get {} one2one item'.format(df_homo.shape[0]))

adata_ref = adata_ref[:, df_homo['gn_ref']].copy()
adata_que = adata_que[:, df_homo['gn_que']].copy()
adata_que.var.index = df_homo['gn_ref'].to_numpy()

[msg] get 14859 one2one item


In [6]:
# TOSICA.train(ref_adata, gmt_path,project=<my_project>,label_name=<label_key>)
p_out = p_root_model.joinpath('TOSICA_LC_01_test')

## Step 1: Training the model

In [7]:
# TOSICA.train(adata_ref, str(p_gmt),p_out,
#              label_name='sub_cell_type')

## Step 2: Prediect by the model

In [8]:
def get_TOSICA_weight_path(p):
    df_info = ut.df.iter_dir(p, path_match='model*pth')
    df_info['epoch'] = df_info['name'].str.extract("model-(\\d+)\\.pth", expand=False).astype(float)
    df_info = df_info.sort_values('epoch', ascending=False).reset_index(drop=True)
    print('[msg] get max epoch {}'.format(df_info.at[0, 'epoch']))
    return df_info.at[0, 'path']

In [9]:
p_weight = get_TOSICA_weight_path(p_out.joinpath('TrainingModel'))

new_adata = TOSICA.pre(adata_que,model_weight_path=p_weight,project=p_out)
new_adata.obs = new_adata.obs.loc[:, 'Prediction,Probability'.split(',')]

[msg] get max epoch 9.0
cpu
0
646


/public/workspace/licanchengup/apps/miniconda3/envs/TOSICA/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [10]:
new_adata

AnnData object with n_obs × n_vars = 646 × 299
    obs: 'Prediction', 'Probability'